# Sentiment Analysis - Phase 1 (Baseline Models)

## Goal
Train a fast and effective **Linear Support Vector Machine (SVM)** for sentiment analysis.

## Simplified Workflow
1.  Load Data
2.  Clean Text
3.  Vectorize (TF-IDF)
4.  Train SVM
5.  Save Model


In [ ]:
# 1. Setup Environment
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
import re
import string
import nltk
from nltk.corpus import stopwords
import joblib
import os

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    print("Downloading stopwords...")
    nltk.download('stopwords')


## 2. Load Data

In [ ]:
if os.path.exists('train_data.csv') and os.path.exists('test_data.csv'):
    print("Loading Train Data...")
    train_df = pd.read_csv('train_data.csv')
    print(f"Loaded Train Data: {len(train_df):,} rows.")
    
    print("Loading Test Data...")
    test_df = pd.read_csv('test_data.csv')
    print(f"Loaded Test Data: {len(test_df):,} rows.")
else:
    print("Error: CSV files not found! Please make sure 'train_data.csv' and 'test_data.csv' are in the same folder.")

## 3. Preprocessing

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(f"[{string.punctuation}]", "", text)
    words = text.split()
    stop_words = set(stopwords.words('english'))
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

if 'clean_text' not in train_df.columns:
    print("Cleaning Training Data...")
    train_df['clean_text'] = train_df['sentence'].apply(clean_text)
else:
    print("Training data already cleaned.")

if 'clean_text' not in test_df.columns:
    print("Cleaning Test Data...")
    test_df['clean_text'] = test_df['sentence'].apply(clean_text)
else:
    print("Test data already cleaned.")

## 4. Train SVM Model

In [ ]:
print("Vectorizing...")
vectorizer = TfidfVectorizer(max_features=5000)
X_train = vectorizer.fit_transform(train_df['clean_text'])
y_train = train_df['sentiment']
X_test = vectorizer.transform(test_df['clean_text'])
y_test = test_df['sentiment']

print("Training Linear SVM...")
svm_model = LinearSVC(dual=False, random_state=42)
svm_model.fit(X_train, y_train)

print("Evaluating...")
y_pred = svm_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"SVM Accuracy: {acc*100:.2f}%")

## 5. Save Artifacts

In [ ]:
# Create directory
if not os.path.exists('saved_models'):
    os.makedirs('saved_models')

print("Saving Vectorizer...")
joblib.dump(vectorizer, 'saved_models/tfidf_vectorizer.pkl')

print("Saving SVM Model...")
joblib.dump(svm_model, 'saved_models/svm_model.pkl')

print("Done! Models saved to 'saved_models/' directory.")